# 03 - Generation and Verifier Smoke Test

This notebook checks the full demo pipeline after the data and retrieval notebooks: `llm`, `rag`, and `verified` modes.

The first complete run showed that the pipeline works mechanically and the model answers in Turkish, but answer quality is strongly limited by retrieval quality. A cited RAG answer can still be wrong if the retrieved passages are about the wrong legal issue.


## Before Running

This notebook uses the actual backend pipeline instead of rebuilding retrieval inside the notebook. Before running the main cells:

1. Build the index with `python scripts/build_index.py --limit 2000` for a quick run, or without `--limit` for the full corpus.
2. Start Ollama if the config uses the local backend: `ollama serve`.
3. Make sure the model in `configs/default.yaml` is pulled locally, for example `ollama pull qwen2.5:7b-instruct`.

The local Ollama model used in the first run was `qwen2.5:7b-instruct`. On this machine, Ollama stores it under `/Users/onur/.ollama/models`, while Hugging Face embedding models are cached under `/Users/onur/.cache/huggingface/hub`.


In [11]:
# Install these if the notebook environment does not have them yet:
# %pip install -e ..
# %pip install pandas pydantic pyyaml fastapi huggingface-hub sentence-transformers faiss-cpu


In [12]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import pandas as pd


In [13]:
def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "rag_turkish_law").exists():
            return candidate.resolve()
    raise RuntimeError("Could not find repo root. Open this notebook from the project workspace.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RESULTS_DIR = ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = ROOT / "data" / "processed" / "faiss.index"
META_PATH = ROOT / "data" / "processed" / "passage_meta.jsonl"

print("ROOT:", ROOT)
print("INDEX_PATH exists:", INDEX_PATH.exists())
print("META_PATH exists:", META_PATH.exists())


ROOT: /Users/onur/Desktop/CS455/Term Project
INDEX_PATH exists: True
META_PATH exists: True


## Load Project Pipeline

The model and backend settings come from `configs/default.yaml`, so this notebook stays aligned with the demo app.

For the first run, the important settings were local Ollama generation, `qwen2.5:7b-instruct`, `intfloat/multilingual-e5-base` retrieval embeddings, `top_k = 5`, and reranking disabled.


In [14]:
from rag_turkish_law.config import load_config
from rag_turkish_law.api.pipeline import run_pipeline

cfg = load_config()
print("generation provider:", cfg.generation.provider)
print("generation model:", cfg.generation.hf_model)
print("retrieval model:", cfg.retrieval.embedding_model)
print("top_k default:", cfg.retrieval.top_k)
print("rerank enabled:", cfg.retrieval.rerank.enabled)


generation provider: ollama
generation model: qwen2.5:7b-instruct
retrieval model: intfloat/multilingual-e5-base
top_k default: 5
rerank enabled: False


## Questions For The Smoke Test

The questions are intentionally mixed. Some are normal legal questions, and some are ambiguous, unsupported, future-looking, or risky.

This small set already exposed useful behavior: unsupported/future-law questions were usually refused, urgent case-specific advice was flagged as risky, and natural rental/inheritance questions showed retrieval weaknesses.


In [15]:
questions = [
    {
        "id": "rent_early_exit",
        "question": "Kira sözleşmesi bitmeden kiracı evden çıkabilir mi?",
        "type": "normal",
        "what_to_watch": "Cevap kaynak gösteriyor mu ve fazla somut hukuki tavsiye vermekten kaçınıyor mu?",
    },
    {
        "id": "severance",
        "question": "İşten çıkarılan bir işçi hangi şartlarda kıdem tazminatı alabilir?",
        "type": "normal",
        "what_to_watch": "Getirme aşaması iş hukuku ile ilgili pasajları bulabiliyor mu?",
    },
    {
        "id": "inheritance_house",
        "question": "Miras kalan ev kardeşler arasında nasıl paylaşılır?",
        "type": "normal",
        "what_to_watch": "Model genel bilgiyi hukuki prosedürden ayırabiliyor mu?",
    },
    {
        "id": "urgent_specific_advice",
        "question": "Komşumla kavga ettim, bugün hemen dava açmam gerekir mi?",
        "type": "risk_or_ambiguous",
        "what_to_watch": "Cevap veya doğrulayıcı bunun somut olaya özel bir soru olduğunu yakalıyor mu?",
    },
    {
        "id": "unsupported_policy",
        "question": "Türkiye'de 2027 yılında yürürlüğe girecek yeni kira kanunu ne diyor?",
        "type": "unsupported_or_future",
        "what_to_watch": "Sistem kaynak yokken reddediyor mu, yoksa cevap uyduruyor mu?",
    },
]

pd.DataFrame(questions)


,id,question,type,what_to_watch
0,rent_early_exit,Kira sözleşmesi bitmeden kiracı evden çıkabili...,normal,Cevap kaynak gösteriyor mu ve fazla somut huku...
1,severance,İşten çıkarılan bir işçi hangi şartlarda kıdem...,normal,Getirme aşaması iş hukuku ile ilgili pasajları...
2,inheritance_house,Miras kalan ev kardeşler arasında nasıl paylaş...,normal,Model genel bilgiyi hukuki prosedürden ayırabi...
3,urgent_specific_advice,"Komşumla kavga ettim, bugün hemen dava açmam g...",risk_or_ambiguous,Cevap veya doğrulayıcı bunun somut olaya özel ...
4,unsupported_policy,Türkiye'de 2027 yılında yürürlüğe girecek yeni...,unsupported_or_future,"Sistem kaynak yokken reddediyor mu, yoksa ceva..."


## Run One Question First

Running one question before the full batch helps catch missing index files, model-serving problems, or prompt issues before spending time on all modes.

In the first run, the model answered in Turkish, but the rental-contract question retrieved mostly `konut dokunulmazlığı` passages. That made the generated answer look grounded even though the grounding was legally off-topic.


In [16]:
if not INDEX_PATH.exists() or not META_PATH.exists():
    print("Index files are missing. Run `python scripts/build_index.py --limit 2000` from the repo root first.")
else:
    sample_question = questions[0]["question"]
    response = run_pipeline(sample_question, mode="rag", k=5)
    print("SORU:", response.question)
    print()
    print("CEVAP:")
    print(response.answer)
    print()
    print("KAYNAKLAR")
    for source in response.sources[:3]:
        print(f"- {source.id} score={source.score:.3f} title={source.title[:120]}")


SORU: Kira sözleşmesi bitmeden kiracı evden çıkabilir mi?

CEVAP:
Kira sözleşmesi bitmeden kiracı evden çıkabilir, ancak bu durumda konut ihlali olabilir [2]. Kiracı rızası hilafına konuttan çıkmak isteyebilir, ancak bu durumu gerçekleştirmek için konut sahibinin rızasını alması gereklidir [1]. Kiracı rızasız olarak konutta kalma kararı verirse, konut sahibi ihlalin cezasına maruz kalmaya risk altına alabilir [2][3].

KAYNAKLAR
- P-000994 score=0.835 title=Konut sahibinin rızası hilafına konuttan çıkmamanın anlamı nedir?
- P-000995 score=0.823 title=Konut ihlalinin rızayla girilmesi durumunda hangi şartlar altında suç oluşur?
- P-000049 score=0.819 title=Davalı, malın zilyetliğini sürdürmek için hangi tür belgeleri kanıt olarak sunabilir?


## Batch Run Across Modes

This cell runs each question through `llm`, `rag`, and `verified`. It can take a while because `verified` calls both the generator and the verifier.

The first run produced 15 records with no runtime errors. Local latency was reasonable for a smoke test: RAG calls were usually a few seconds, and verified calls took longer because of the verifier step.


In [17]:
MODES = ["llm", "rag", "verified"]
TOP_K = 5

records = []

if not INDEX_PATH.exists() or not META_PATH.exists():
    print("Skipping batch run because the index files are missing.")
else:
    for item in questions:
        for mode in MODES:
            print(f"Çalışıyor: {item['id']} / {mode}...")
            started = time.perf_counter()
            try:
                response = run_pipeline(item["question"], mode=mode, k=TOP_K)
                elapsed_ms = int((time.perf_counter() - started) * 1000)
                data = response.model_dump()
                data.update(
                    {
                        "question_id": item["id"],
                        "question_type": item["type"],
                        "what_to_watch": item["what_to_watch"],
                        "elapsed_ms_total": elapsed_ms,
                        "error": None,
                    }
                )
            except Exception as exc:  # keep going so one model failure does not lose the batch
                elapsed_ms = int((time.perf_counter() - started) * 1000)
                data = {
                    "question_id": item["id"],
                    "question_type": item["type"],
                    "what_to_watch": item["what_to_watch"],
                    "question": item["question"],
                    "mode": mode,
                    "answer": "",
                    "llm_only": None,
                    "sources": [],
                    "verdict": None,
                    "timings": [],
                    "elapsed_ms_total": elapsed_ms,
                    "error": repr(exc),
                }
            records.append(data)

print("records:", len(records))


Çalışıyor: rent_early_exit / llm...
Çalışıyor: rent_early_exit / rag...
Çalışıyor: rent_early_exit / verified...
Çalışıyor: severance / llm...
Çalışıyor: severance / rag...
Çalışıyor: severance / verified...
Çalışıyor: inheritance_house / llm...
Çalışıyor: inheritance_house / rag...
Çalışıyor: inheritance_house / verified...
Çalışıyor: urgent_specific_advice / llm...
Çalışıyor: urgent_specific_advice / rag...
Çalışıyor: urgent_specific_advice / verified...
Çalışıyor: unsupported_policy / llm...
Çalışıyor: unsupported_policy / rag...
Çalışıyor: unsupported_policy / verified...
records: 15


## Summary Table

The summary table is mainly for checking whether the run completed, how many sources were returned, what verifier verdicts appeared, and roughly how long each step took.

In the first run, the verifier returned useful high-level labels for several cases: `insufficient` for unsupported/future-law style questions, `risk` for the urgent neighbor-dispute question, and `partial` for some weakly grounded answers.


In [18]:
def build_summary_df(records: list[dict]) -> pd.DataFrame:
    def summarize_record(record: dict) -> dict:
        verdict = record.get("verdict") or {}
        timings = {
            t.get("id"): t.get("ms")
            for t in record.get("timings", [])
            if isinstance(t, dict)
        }
        return {
            "question_id": record.get("question_id"),
            "mode": record.get("mode"),
            "type": record.get("question_type"),
            "error": record.get("error"),
            "answer_chars": len(record.get("answer") or ""),
            "num_sources": len(record.get("sources") or []),
            "verdict": verdict.get("key"),
            "verdict_score": verdict.get("score"),
            "risk": verdict.get("risk"),
            "retrieve_ms": timings.get("retrieve"),
            "generate_ms": timings.get("generate"),
            "verify_ms": timings.get("verify"),
            "elapsed_ms_total": record.get("elapsed_ms_total"),
        }

    return pd.DataFrame([summarize_record(r) for r in records])


if "records" not in globals():
    raise RuntimeError("Run the batch cell first so `records` exists.")

summary_df = build_summary_df(records)
summary_df


,question_id,mode,type,error,answer_chars,num_sources,verdict,verdict_score,risk,retrieve_ms,generate_ms,verify_ms,elapsed_ms_total
0,rent_early_exit,llm,normal,None,363,0,NaN,NaN,NaN,NaN,2389,NaN,2389
1,rent_early_exit,rag,normal,None,232,5,NaN,NaN,NaN,31.0,3078,NaN,3109
2,rent_early_exit,verified,normal,None,328,5,partial,0.666667,low,21.0,2109,4591.0,6722
3,severance,llm,normal,None,297,0,NaN,NaN,NaN,NaN,2099,NaN,2099
4,severance,rag,normal,None,63,5,NaN,NaN,NaN,19.0,2178,NaN,2198
5,severance,verified,normal,None,63,5,insufficient,0.000000,medium,24.0,537,2450.0,3012
6,inheritance_house,llm,normal,None,211,0,NaN,NaN,NaN,NaN,1540,NaN,1540
7,inheritance_house,rag,normal,None,249,5,NaN,NaN,NaN,25.0,2958,NaN,2984
8,inheritance_house,verified,normal,None,230,5,partial,0.500000,low,27.0,1524,4389.0,5942
9,urgent_specific_advice,llm,risk_or_ambiguous,None,210,0,NaN,NaN,NaN,NaN,1449,NaN,1449


## Inspect Answers And Sources

This is the most important part of the notebook. The CSV summary can say that the pipeline ran, but only reading the answer and top sources shows whether the system is actually reliable.

Main finding from the first run: retrieval errors can turn into confident, cited generation errors. The rental-contract example is the clearest case.


In [19]:
def show_record(question_id: str, mode: str) -> None:
    matches = [r for r in records if r.get("question_id") == question_id and r.get("mode") == mode]
    if not matches:
        print("Eşleşen kayıt yok.")
        return

    record = matches[0]
    print("SORU:", record.get("question"))
    print("MOD:", record.get("mode"))
    print("HATA:", record.get("error"))
    print()
    print("CEVAP")
    print(record.get("answer"))

    print()
    print("KAYNAKLAR")
    for source in record.get("sources", [])[:5]:
        score = source.get("score")
        score_text = f"{score:.3f}" if isinstance(score, (int, float)) else str(score)
        print(f"- {source.get('id')} score={score_text} title={source.get('title')}")
        print(" ", (source.get("snippet") or "")[:240])

    verdict = record.get("verdict")
    if verdict:
        print()
        print("KARAR", verdict.get("key"), "score=", verdict.get("score"), "risk=", verdict.get("risk"))
        for claim in verdict.get("claims", [])[:8]:
            print(f"- [{claim.get('status')}] src={claim.get('src')} {claim.get('text')}")

show_record("rent_early_exit", "verified")


SORU: Kira sözleşmesi bitmeden kiracı evden çıkabilir mi?
MOD: verified
HATA: None

CEVAP
Kira sözleşmesi bitmeden kiracı evden çıkabilir, ancak bu durumda konut ihlali olabilir [2]. Kiracı rızası hilafına konuttan çıkmak isteyebilir, ancak bu durumu gerçekleştirmek için konut sahibinin rızasını alması gereklidir [1]. Kiracı rızasız olarak konutta kalma suçu taşıyabilir [2], bu nedenle kiracı evden çıkması gerekir.

KAYNAKLAR
- P-000994 score=0.835 title=Konut sahibinin rızası hilafına konuttan çıkmamanın anlamı nedir?
  Konut sahibinin rızası hilafına konuttan çıkmamak, konuta girmek için rıza vermeye hak sahibi olan kişinin söz, hareket ve tavırlarıyla çıkmaya davet etmesine rağmen failin rızayla girilen yerden ayrılmamasıdır.
- P-000995 score=0.823 title=Konut ihlalinin rızayla girilmesi durumunda hangi şartlar altında suç oluşur?
  Konut ihlalinin rızayla girilmesi durumunda, konuta rıza ile girildikten sonra konuttan çıkmamak şartıyla suç oluşur.
- P-000049 score=0.819 title=Daval

## Save Smoke-Test Outputs

The JSONL file keeps the full responses, including answers, sources, timings, and verifier claims. The CSV keeps the compact summary table.

These outputs should be used as error-analysis material before we make implementation fixes.


In [20]:
if "records" not in globals():
    raise RuntimeError("Run the batch cell first so `records` exists.")

if "summary_df" not in globals():
    if "build_summary_df" not in globals():
        def build_summary_df(records: list[dict]) -> pd.DataFrame:
            rows = []
            for record in records:
                verdict = record.get("verdict") or {}
                timings = {
                    t.get("id"): t.get("ms")
                    for t in record.get("timings", [])
                    if isinstance(t, dict)
                }
                rows.append(
                    {
                        "question_id": record.get("question_id"),
                        "mode": record.get("mode"),
                        "type": record.get("question_type"),
                        "error": record.get("error"),
                        "answer_chars": len(record.get("answer") or ""),
                        "num_sources": len(record.get("sources") or []),
                        "verdict": verdict.get("key"),
                        "verdict_score": verdict.get("score"),
                        "risk": verdict.get("risk"),
                        "retrieve_ms": timings.get("retrieve"),
                        "generate_ms": timings.get("generate"),
                        "verify_ms": timings.get("verify"),
                        "elapsed_ms_total": record.get("elapsed_ms_total"),
                    }
                )
            return pd.DataFrame(rows)
    summary_df = build_summary_df(records)

jsonl_path = RESULTS_DIR / "generation_verifier_smoke_results.jsonl"
csv_path = RESULTS_DIR / "generation_verifier_smoke_summary.csv"

with jsonl_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + chr(10))

summary_df.to_csv(csv_path, index=False)

print("wrote", jsonl_path)
print("wrote", csv_path)


wrote /Users/onur/Desktop/CS455/Term Project/evaluation/results/generation_verifier_smoke_results.jsonl
wrote /Users/onur/Desktop/CS455/Term Project/evaluation/results/generation_verifier_smoke_summary.csv


## Notes After Running

First-run findings:

- The full pipeline runs end-to-end without model/runtime errors.
- The model answers in Turkish with the current prompts.
- Retrieval is the main bottleneck on natural questions.
- Strong seed retrieval metrics from notebook 02 did not transfer to all natural smoke-test questions.
- The verifier is useful for surfacing `risk`, `partial`, and `insufficient` cases, but it does not fully solve bad retrieval or weak citation grounding.

Concrete examples:

- `rent_early_exit`: retrieved `konut dokunulmazlığı` passages for a rental-contract question, leading to a bad cited answer.
- `severance`: retrieved general compensation passages and correctly ended up with an insufficient-style answer.
- `urgent_specific_advice`: verifier marked the answer as high risk, which is the kind of behavior we want.
- `unsupported_policy`: the system mostly refused because the sources did not cover the future-law claim.

Next code fixes should focus on manual/adversarial evaluation wiring, actual rerank ablation, retrieval confidence/coverage gating, and citation-aware verification.
